# Issue #4: Clean Movie Keywords

Runs `models\cleanKeywords.py` to read `data\raw\keywords.csv` and output `data\processed\cleaned_keywords.csv`

`cleanKeywords.py`: 
- removes empty or duplicated rows.
- parses JSON-like keywords column, extracts keywords and normalizes them.
- outputs the movie id and json like keywords column.

## 0: Setup

In [ ]:
import pandas as pd
import numpy as np
import ast
import json
import re

from pathlib import Path

# Project root
PROJECT_ROOT = Path(__file__).resolve().parents[1]

# Input and output csv files
input_file = PROJECT_ROOT / "data" / "raw" / "keywords.csv"
output_file = PROJECT_ROOT / "data" / "processed" / "cleaned_keywords.csv"

# Create the processed folder if it does not already exist
output_file.parent.mkdir(parents=True, exist_ok=True)

## 1: Load and Inspect Dataset

In [ ]:
# Read keywords.csv
df = pd.read_csv(input_file)
df = df[["id", "keywords"]]

# Treat blank values as missing
df["id"] = df["id"].replace(r"^\s*$", np.nan, regex=True)
df["keywords"] = df["keywords"].replace(r"^\s*$", np.nan, regex=True)

# Inspect dataset
missing_rows = df.isna().any(axis=1).sum()
duplicate_rows = df.duplicated().sum()

print("\n--- Dataset Inspection ---")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"Dataset dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Rows with missing values: {missing_rows}")
print(f"Duplicate rows: {duplicate_rows}")

print("\nData types:")
print(df.dtypes)
print("--------------------------\n")

**Dataset**

Total rows: 46419

Total columns: 2

Dataset dimensions: 46419 rows x 2 columns

Rows with missing values: 0

Duplicate rows: 987


**Data types:**

id          int64

keywords      str

dtype: object

## 2: Remove Missing and Duplicate Rows

In [ ]:
df = df.dropna(subset=["id", "keywords"])
df = df.drop_duplicates()

## 3: Parse and Normalize Keywords

In [ ]:
def clean_keywords(value):
    try:
        keyword_data = ast.literal_eval(value)
        cleaned = []

        for item in keyword_data:
            if isinstance(item, dict) and "name" in item:
                keyword = item["name"].lower()
                keyword = re.sub(r"\s+", " ", keyword).strip()

                if keyword and keyword not in cleaned:
                    cleaned.append(keyword)

        return cleaned

    except (ValueError, SyntaxError, TypeError):
        return []

## 4: Extract Cleaned Keywords into Consistent Format

In [ ]:
df["keywords"] = df["keywords"].apply(clean_keywords)

# Remove movies with no usable keywords
df = df[df["keywords"].map(len) > 0]

df["_original_order"] = np.arange(len(df))
# Ensure one row per movie
df = (
    df.sort_values("_original_order")
      .groupby("id", sort=False, as_index=False)["keywords"]
      .agg(lambda lists: list(dict.fromkeys(
          keyword
          for keyword_list in lists
          for keyword in keyword_list
      )))
)

## 5: Format and Export the Cleaned Dataset

In [ ]:
# Convert Python lists into valid JSON arrays
df["keywords"] = df["keywords"].apply(json.dumps)

# Rename the movie ID column
df = df.rename(columns={"id": "movie_id"})

# Save the cleaned file
df.to_csv(output_file, index=False)

print("Created cleaned_keywords.csv")
print(df.head())